# 01 Data Loading — UC RUSAL

Загрузка данных из `rusal_complete_v4.xlsx` в `data_mart_v2.db`.

**Два этапа:**
1. ExcelLoader — IS/BS/CF + PPE + Debt + Segments + Macro + KPI
2. Schedule Loader — Intangibles + Tax + Provisions + Associates + Operational Drivers


In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT))
COMPANY = 'rusal'
EXCEL = ROOT / f'companies/{COMPANY}/data/rusal_complete_v4.xlsx'
DB = ROOT / 'data_mart_v2.db'

print(f'ROOT:  {ROOT}')
print(f'Excel: {EXCEL.name}')
print(f'DB:    {DB.name}')


## 1. ExcelLoader — Face Sheets + Canonical


In [ ]:
from engine.loader.excel import ExcelLoader
from engine.database.repository import Repository

with Repository(str(DB)) as repo:
    loader = ExcelLoader(
        company_id=COMPANY,
        repo=repo,
        db_unit='USD',
        input_default_unit='mUSD',
    )
    result = loader.load(EXCEL)

print(f'Rows written: {result.rows_written}')
print(f'Errors: {result.errors[:3]}')
print(f'Warnings: {result.warnings[:3]}')


## 2. Schedule Loader — Notes Data


In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, str(ROOT / 'tools/load_schedule_sheets.py'),
     '--company', COMPANY,
     '--excel', str(EXCEL),
     '--db', str(DB)],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])


## 3. Verification


In [ ]:
import sqlite3

with sqlite3.connect(str(DB)) as conn:
    TABLES = [
        ('history_is', 'company_id'),
        ('history_bs', 'company_id'),
        ('history_cf', 'company_id'),
        ('ppe_components', 'company_id'),
        ('intangible_assets', 'company_id'),
        ('tax_schedule', 'company_id'),
        ('provisions_schedule', 'company_id'),
        ('associates_schedule', 'company_id'),
        ('operational_drivers', 'company'),
        ('debt_instruments', 'company_id'),
    ]
    print(f'{"Table":30} {"Rows":>6}')
    print('-' * 38)
    total = 0
    for tbl, cc in TABLES:
        try:
            cnt = conn.execute(f'SELECT COUNT(*) FROM {tbl} WHERE {cc}=?',
                              (COMPANY,)).fetchone()[0]
            total += cnt
            print(f'  {tbl:28} {cnt:>6}')
        except Exception as e:
            print(f'  {tbl:28} ERROR')
    print(f'{"TOTAL":>30} {total:>6}')


In [ ]:
# Quick model check
import logging; logging.disable(logging.CRITICAL)
from engine.orchestrator import build_model

r = build_model(COMPANY, run_preprocessor=False, run_model=True)
bs = max(r.model_result.bs_diffs.values())
print(f'{COMPANY}: BS={bs:.6f}')
assert bs < 0.001, f'Model broken! BS={bs}'
print('\u2705 Model OK')
